<h1 style="text-align: center; font-weight: bold;">H&M经营分析与客户分层运营</h1>
<hr>

# 1. 项目背景与数据集介绍

## 1.1 业务背景

H&M集团（H&M Hennes & Mauritz AB）是创立于瑞典斯德哥尔摩的全球性快时尚零售企业，成立于1947年，目前在斯德哥尔摩纳斯达克交易所上市，旗下拥有H&M、COS、Weekday、ARKET等多个品牌。H&M的核心经营理念是"以最优惠的价格提供时尚和品质"——通过将制造外包给低成本国家工厂控制成本、缩短设计到上架的周期、融合北欧简约美学与流行元素，定位中档快时尚，主要面向学生及年轻职场人士。截至2026年5月，H&M 品牌在81个市场设有3,599家线下门店，在62个市场提供线上购物服务。

本项目进行筛选处理后使用的Kaggle H&M数据集涵盖2019年9月至2020年8月的交易、用户与商品信息。这一时间窗口对H&M而言具有特殊意义，2019年年报披露H&M已在日益融合数字渠道与实体渠道，据集团2020年年报，2020年净减少了58家门店。窗口横跨疫情前与疫情爆发期，是观察H&M在线上线下渠道格局关键调整期、突发外部冲击下业务表现、客群行为、品类偏好与留存表现的难得样本。

尽管数据距今已有6年，H&M集团仍延续其核心战略与客群定位，根据2026年半年报，截至2026年5月31日H&M集团门店总数为4,038家，较上年同期减少128家（约3%），并计划于2026年新开约90家、关闭约170家门店，通过开店、关店与改建持续推进门店组合的优化；与此同时，H&M品牌线下门店仍覆盖81个市场，多于线上62个市场。这一现状与本项目后续基于数据集提出的渠道建议方向一致——即以线上作为主要销售渠道持续投入，同步筛选优化线下门店结构、保留具备社交体验价值与风险对冲功能的门店。因此，基于该窗口得出的客户分层、品类偏好、留存模式等洞察，对理解该品牌当下运营仍具有参考价值。

## 1.2 数据集介绍

本数据集选自Kaggle在 2022 年举办的"H&M Personalized Fashion Recommendations"竞赛，原始数据由H&M提供，为真实历史交易记录。数据集包含Customers、Articles、Transactions三张表，可通过customer_id与article_id 字段串联。三张表的体量分别为：articles表约105,542条商品信息、customers表约1,371,980条顾客记录（去重后共988,865位独立顾客）、transactions表约3,178万条交易记录，时间跨度为2018年9月20日至2020年9月22日。

考虑到分析效率，本项目截取其在2019年9月1日至2020年8月31日的交易记录，从总顾客中随机抽样20%（共 197,773 名顾客），同步筛选这些顾客产生的交易记录并作为分析样本。为优化运行性能，对三张表中低分析价值的字段进行剔除，并将object类型字段转为category格式以节约内存。

关于价格字段的处理：Kaggle原始数据集的price字段为归一化值（0-1区间，无法直接反映真实价格。本项目通过获取淘宝 H&M 官方旗舰店120件T恤的当前售价，与数据集中T恤品类的归一化均价对比，求出还原系数，将其应用于全表price字段生成estimated_price字段。该方法基于T恤品类的价格关系外推至全部品类，存在一定假设偏差，所有金额相关的分析结果建议作趋势性参考而非绝对值。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')


from matplotlib.ticker import PercentFormatter

import gc

In [ ]:
from pathlib import Path

DATA_DIR = Path('../data')

articles = pd.read_csv(DATA_DIR / 'articles.csv')
customers = pd.read_csv(DATA_DIR / 'customers.csv')
transactions = pd.read_csv(DATA_DIR / 'transactions_train.csv')

In [ ]:
print(f'articles:')
print(articles.info())
print(f'customers:')
print(customers.info())
print(f'transactions:')
print(transactions.info())

In [ ]:
print(transactions['t_dat'].min())
print(transactions['t_dat'].max())

In [ ]:
# 由于数据集最后一月数据不全，并出于节约内存的需要，这里将选择2019.9-2020.8内的数据作为分析范围
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
transactions = transactions[(transactions['t_dat'] >= '2019-09-01')&(transactions['t_dat'] <= '2020-08-31')]
transactions.reset_index(drop=True, inplace=True)
gc.collect()

In [ ]:
all_customers = transactions['customer_id'].unique()

# 由于电脑性能压力，这里选择随机抽样20%的顾客，后续针对这些顾客产生的交易记录进行分析
np.random.seed(42)
keep_customers = np.random.choice(all_customers, size=int(len(all_customers)*0.2), replace=False)
transactions = transactions[transactions['customer_id'].isin(keep_customers)].reset_index(drop=True)
customers = customers[customers['customer_id'].isin(keep_customers)].reset_index(drop=True)

gc.collect()

print(len(customers))
print(len(transactions))

# 2.数据处理

## 2.1 customers表

In [ ]:
# 数据概览
customers.head(1)

postal_code字段为脱敏后的哈希值，无法还原为真实地理信息，属无效字段

In [ ]:
# 检查数据缺失率
customers.isnull().sum()/len(customers)

FN、Active字段缺失率均超过60%，填充后真实率存疑，且与分析关联度不高，属无效字段

In [ ]:
# 检查数据
cols = ['club_member_status', 'fashion_news_frequency', 'age']
for col in cols:
    print(f'\n{col}')
    print(customers[col].unique().tolist())
    print(customers[col].describe())

club_member_status中ACTIVE占比高达96%、fashion_news_frequency中None占比63%，占比高度倾斜，难以支撑有效的分群分析，属无效字段

In [ ]:
# 删除无效字段并检查重复记录
customers = customers.drop(columns=['FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'postal_code']) 
customers[customers.duplicated()]

In [ ]:
# 修改字段格式
customers['customer_id'] = customers['customer_id'].astype('category') #减少内存
customers['age'] = customers['age'].astype('Int64') #保留缺失值并显示整数
customers.head(1)

年龄分层

In [ ]:
print(customers['age'].value_counts().sort_index().tail(10))

In [ ]:
bins = [15, 25, 35, 45, 55, 65, 99]
labels = ['16-25岁', '26-35岁', '36-45岁', '46-55岁', '56-65岁', '66岁及以上']
customers['age_group'] = pd.cut(customers['age'], bins=bins, labels=labels, right=True)
customers.head(1)

In [ ]:
customers.info()

## 2.2 articles表

In [ ]:
articles.head(1)

In [ ]:
# 检查数据
articles.columns.tolist()

存在大量'_no'、'_id'商品编码类字段为冗余

In [ ]:
col = ['prod_name', 'product_type_name', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'perceived_colour_master_name', 'department_name', 'index_name', 'index_group_name', 'section_name', 'garment_group_name']
for i in col:
    print(f'\n{i} (nunique={articles[i].nunique()}):')
    print(articles[i].value_counts().head(10))

众多'_name'格式的字段颗粒度过细或超出本项目分析范畴，需剔除，最终仅保留'article_id'、'product_type_name'、'index_group_name'字段

In [ ]:
# 删除无用列
articles = articles[['article_id', 'product_type_name', 'index_group_name']]

# 查找缺失值及重复值
print(articles.isnull().sum())
articles[articles.duplicated()]

In [ ]:
# 调整字段格式减少内存负担
articles.info()

In [ ]:
for i in articles.columns:
    if articles[i].dtype == 'object':
        articles[i] = articles[i].astype('category')
        
articles.info()

## 2.3 transactions表

In [ ]:
# 调整字段格式
transactions['customer_id'] = transactions['customer_id'].astype('category')
transactions['sales_channel_id'] = transactions['sales_channel_id'].astype('int8')
transactions.info()

In [ ]:
# 检查重复购买记录
g = transactions.groupby(['t_dat','customer_id','article_id'], observed=True).size()
print(g.value_counts().sort_index().head(10))
print((g>5).sum(), (g>5).sum() / len(g))

购买单件占比90.7%，超过5件的仅占万分之6.5，最大121件，属于正常购买衰减形态。极端值可能来自批发或团购，占比极低，予以保留不做截断。

In [ ]:
# 检查检查外键完整性、缺失值
notin_customer_id = transactions[~transactions['customer_id'].isin(customers['customer_id'].unique())]
print(f'transactions 中不在 customers 表的记录数: {len(notin_customer_id)}')
notin_article_id = transactions[~transactions['article_id'].isin(articles['article_id'].unique())]
print(f'transactions 中不在 articles 表的记录数: {len(notin_article_id)}')

transactions.isnull().sum()

In [ ]:
# 增加月份列及季节列
transactions['year_month'] = transactions['t_dat'].dt.to_period('M')
transactions['month'] = transactions['t_dat'].dt.month

conditions = [
    transactions['month'].isin([3, 4, 5]),
    transactions['month'].isin([6, 7, 8]),
    transactions['month'].isin([9, 10, 11]),
]

transactions['season'] = np.select(conditions, ['Spring', 'Summer', 'Fall'], default='Winter')

In [ ]:
print(customers.shape)
print(articles.shape)
print(transactions.shape)

# 3. 探索性数据分析（EDA）

## 3.1 单变量分析

### 3.1.1 价格分布

In [ ]:
#为便于可读性，调整数值显示小数点后五位
pd.set_option('display.float_format', lambda x: f'{x:.5f}')
transactions['price'].describe()

In [ ]:
T = articles[articles['product_type_name']=='T-shirt']
T_tran = pd.merge(transactions, T, how='inner', left_on = 'article_id', right_on = 'article_id')
T_mean = T_tran['price'].mean()
#经H&M淘宝官旗获取到的120款T恤价格计算得出T恤平均价格大约在88元左右，获取系数
k = 88/T_mean
print(k)
print(T_tran.shape)

In [ ]:
transactions['estimated_price'] = round(transactions['price']*k, 2)
transactions['estimated_price'].describe()

In [ ]:
trans_mon = transactions[['year_month', 'article_id', 'sales_channel_id']]
trans_mon_counts = trans_mon.groupby(['year_month', 'sales_channel_id']).size().unstack(fill_value=0)
trans_mon_counts

关键异常点：2020年4月销售渠道1的销量骤减为0，同时渠道2的销量保持活跃并相比上月大幅增加，结合外部宏观环境，2020年4月全球处于疫情之中，欧洲等地由于居家令限制许多商超线下门店大规模暂时关闭，并且在H&M2020年半年报中提到H&M四月暂时关闭了超过80%的门店，据此推测销售渠道1为线下门店，2为线上渠道。这一推测也符合在后续月份中，当渠道1逐渐恢复销售时渠道2的销量有所回落的现象。

In [ ]:
plt.rcParams['font.sans-serif'] = 'SimHei'
plt.figure(figsize=(15, 6))
#子图1：商品销售价格分布图
plt.subplot(1, 2, 1)
sns.histplot(transactions['estimated_price'], bins=50, color='#E50914', edgecolor='white', linewidth=3)
plt.xlim(0, 800)
plt.title('H&M近一年(2019-2020)商品销售价格分布', fontsize=14, fontweight='bold')
plt.xlabel('商品价格', fontsize=12)
plt.ylabel('销售数量', fontsize=12)
plt.grid(axis='y', alpha=0.3)


#子图2：商品销售价格箱线图
plt.subplot(1, 2, 2)
sns.boxplot(x=transactions['sales_channel_id'], y=transactions['estimated_price'])
plt.title('不同销售渠道价格分布对比', fontsize=15)
plt.xlabel('销售渠道', fontsize=12)
plt.ylim(0, 1000)

plt.tight_layout()
plt.show()

小结：
- H&M商品价格区间高度集中在百元附近，呈现典型的长尾分布，符合快时尚品牌的平价定位。
- 线上渠道的中位价格显著高于线下门店，可能是由于线下受限于物理货架空间、存在更强的清仓压力，需要经常对旧款进行大幅折扣促销，从而拉低了线下均价。
- 同时H&M的部分高端系列开放给线下的销售渠道较少，更多选择在线上销售，因此拉高了线上渠道的中位线，且箱线图中的异常值更密集。另外，呈现这种现象也可能与顾客的消费习惯有关，线下时间有限顾客会更倾向于随手买一件，而线上购物则给了顾客足够长的时间对比挑选，有时线上渠道促销机制（如满减）更易引导顾客进行高客单价的计划性消费。

### 3.1.2 顾客年龄分布

In [ ]:
plt.figure(figsize=(12, 5)) 
#子图1：年龄分布直方图
plt.subplot(1, 2, 1)
sns.histplot(customers['age'].dropna(), bins=20, color='#E50914', edgecolor='white', linewidth=3)
plt.title('顾客年龄分布', fontsize=15, fontweight='bold')
plt.xlabel('顾客年龄', fontsize=12)
plt.ylabel('顾客人数', fontsize=12)
plt.grid(axis='y', alpha=0.3)

#子图2：年龄群体分布
plt.subplot(1, 2, 2)
age_group_counts = customers.groupby('age_group').size().sort_values(ascending=True)
colors = plt.cm.Reds(np.linspace(0, 1, len(age_group_counts.index)))
plt.pie(age_group_counts, labels=age_group_counts.index, autopct='%.1f%%', startangle=90, colors=colors)
plt.title('各年龄阶段顾客占比')

plt.tight_layout()
plt.show()

In [ ]:
pd.reset_option('display.float_format')
customer_transaction = pd.merge(transactions, customers, how='inner', left_on='customer_id', right_on='customer_id')
age_group_analysis = customer_transaction.groupby('age_group')['estimated_price'].agg(
    [('平均消费水平', 'mean'), ('消费金额中位数', 'median'), ('人数', 'count')]
).round(2)
print('各年龄阶段消费能力分析：')
print(age_group_analysis)

In [ ]:
customer_transaction.shape

小结：
- 16-35岁顾客为主要消费人群，占比超过半数，其次为45岁左右的消费群体，占比约30%。呈现“双峰”分布。
- 各年龄群体件均价呈现明显递增趋势，56岁以上群体虽人数占比不高，但对高价商品接受度较高。

### 3.1.3 季节分布

In [ ]:
transactions.head(3)

In [ ]:
year_monthly_sales = transactions.groupby('year_month')['price'].sum().round(2).rename('sales')
monthly_sales = transactions.groupby('month')['price'].sum().round(2).rename('sales')

season_sales = transactions.groupby('season')['price'].sum().round(2).rename('sales')
season_daily_sales = (season_sales/transactions.groupby('season')['t_dat'].nunique()).round(2)
season_daily_sales.sort_values(ascending=True, inplace=True)
season_daily_sales

In [ ]:
plt.figure(figsize=(12, 6), dpi=100)

#子图1：年销售分布
plt.subplot(1, 2, 1)
year_monthly_sales.index = year_monthly_sales.index.astype(str)
plt.plot(year_monthly_sales.index, year_monthly_sales.values, marker='o', color='#E50914')
plt.title('2019-2020月总销售额', fontsize=15)
plt.ylabel('销售金额', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

#子图2：季节销售分布
plt.subplot(1, 2, 2)
colors = plt.cm.Reds(np.linspace(0.1, 0.9, 4))
plt.pie(season_sales, labels=season_sales.index, autopct='%.1f%%', startangle=90, colors=colors, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
plt.title('季节销售占比', fontsize=15)

plt.tight_layout()
plt.show()

小结：
- 从每月的日平均销售额来看，H&M受2020年疫情影响较大，1-2月受季节性淡季与疫情爆发的双重影响跌至了最低点。
- 2020年6月起随疫情情况稍缓而逐渐有所恢复，可能受到走出家门的顾客报复性消费与夏季新品上新、年中大促等影响。
- 全年季节销售较为均匀，秋季略高，可能是因为2019年未受疫情影响拉高了水平。

## 3.2 维度交叉分析

本节将围绕年龄、季节、销售渠道三个核心客户/业务维度，与件均价、人均消费、品类、价格等指标进行交叉分析，识别不同维度下的消费行为差异。

### 3.2.1 年龄维度

#### A. 年龄群体分布与消费能力

综合人数分布、消费贡献和人均消费能力三个维度，可以看到不同年龄群体对 H&M 的价值构成有显著差异。

In [ ]:
cus_tran = (
    customers[['customer_id', 'age_group']]
    .merge(
        transactions[['customer_id', 'estimated_price', 'year_month']], 
        how='inner', on='customer_id'
    )
)

# 计算件单价ASP
age_sales = cus_tran.groupby('age_group')['estimated_price'].sum().round(2)
age_counts = cus_tran.groupby('age_group')['customer_id'].count()
ASP = age_sales/age_counts

# 计算人均消费金额ARPU
customer_counts = cus_tran.groupby('age_group')['customer_id'].nunique()
ARPU = age_sales/customer_counts

In [ ]:
plt.figure(figsize=(12, 12))
HM_RED = '#E50914'

#子图1：不同年龄群体人数分布
plt.subplot(2, 2, 1)
age_size = customers.groupby('age_group')['customer_id'].count()
bars = plt.bar(age_size.index, age_size.values, color=HM_RED)


for bar, val in zip(bars, age_size.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+600, 
             f'{val:,.0f}', ha='center')

plt.title('各年龄群体分布', fontsize=15, fontweight='bold')
plt.xlabel('年龄', fontsize=12)
plt.ylabel('人数', fontsize=12)
plt.grid(axis='y', alpha=0.3)

#子图2：不同年龄阶段消费收入分布
plt.subplot(2, 2, 2)
age_sales_k = age_sales/1000
bars = plt.bar(age_sales_k.index, age_sales_k.values, color=HM_RED)

for bar, val in zip(bars, age_sales_k.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+950, 
             f'{val:,.0f}', ha='center')

plt.title('各年龄群体贡献收益(K)', fontsize=15, fontweight='bold')
plt.xlabel('年龄', fontsize=12)
plt.ylabel('消费金额(K)', fontsize=12)
plt.grid(axis='y', alpha=0.3)

#子图3：不同年龄阶段件单价
plt.subplot(2, 2, 3)

plt.plot(ASP.index, ASP.values, 
         marker='o', color=HM_RED, linewidth=2)
for x, y in zip(ASP.index, ASP.values):
    plt.text(x, y+0.5, f'{y:,.2f}', ha='center')

plt.title('各年龄群体件均价', fontsize=15, fontweight='bold')
plt.xlabel('年龄', fontsize=12)
plt.ylabel('件均价', fontsize=12)
plt.grid(axis='y', alpha=0.3)

#子图4：不同年龄阶段平均消费能力
plt.subplot(2, 2, 4)

plt.plot(ARPU.index, ARPU.values, 
         marker='o', color=HM_RED, linewidth=2)
for x, y in zip(ARPU.index, ARPU.values):
    plt.text(x, y+30, f'{y:,.2f}', ha='center')

plt.title('各年龄群体平均消费能力', fontsize=15, fontweight='bold')
plt.xlabel('年龄', fontsize=12)
plt.ylabel('平均消费能力', fontsize=12)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

小结：
- 根据图表结果，16-25、26-35、46-55三个年龄段是H&M收入的主要来源。除26-35岁外，其余群体的"人数排名"与"收入贡献排名"基本一致，说明对大部分群体来说，人群基数是收入的主要驱动因素。
- 对比左上、右上两张图可以发现一个明显反差——人数排名第二的26-35岁群体，反而成为收益贡献最高的群体。因此26-35岁群体需要单独关注：人数虽低于16-25岁，但人均消费高达3,275，比16-25岁高出54%，是唯一一个以更少人数贡献了更高收益的年龄段，最终贡献了最多收益（约1.7亿元）。
- 26-35岁群体包含大量年轻职场人士，更注重穿衣打扮，买服装频率更高，平均消费能力也最高。因此针对这类群体可偏向推出更多平价且具流行感的当季服饰。
- 综合件均价水平与平均消费能力两张图，件均价随年龄稳定递增（164→196），说明年长群体倾向选择更高价的单品；但人均消费呈倒U型（26-35岁峰值3,276，66岁以上跌至1,793），说明年长群体的"高客单"是建立在"低频次"基础上的，全年总消费反而最低。表明H&M顾客随年龄增长越发注重"少而精"的消费习惯。可针对56岁以上群体推出经典款、耐穿型的高品质单品（如羊毛外套、品质针织、剪裁西装等），契合其"低频但高客单"的购买模式，无需追求高频复购。

In [ ]:
plt.close('all')

#### B. 年龄 × 渠道偏好

在3.1.1可以看到 H&M 整体以线上渠道为主（约 75%）。本节进一步检验这一渠道结构在不同年龄群体间是否存在显著差异。

In [ ]:
cus_chan = customers[['customer_id', 'age_group']].merge(
    transactions[['customer_id', 'sales_channel_id', 'estimated_price']], how='inner', on='customer_id')

age_channel_counts = cus_chan.groupby(['age_group', 'sales_channel_id'])['customer_id'].count().unstack(fill_value=0)
age_channel_counts_pct = age_channel_counts.div(age_channel_counts.sum(axis=1),axis=0)

age_channel_sales = cus_chan.groupby(['age_group', 'sales_channel_id'])['estimated_price'].sum().unstack(fill_value=0)
age_channel_sales_pct = age_channel_sales.div(age_channel_sales.sum(axis=1),axis=0)

age_channel_avg_sales = age_channel_sales/age_channel_counts*100
age_channel_avg_sales_pct = age_channel_avg_sales.div(age_channel_avg_sales.sum(axis=1),axis=0)

cus_chan.shape

In [ ]:
# 提取每个年龄组中"渠道2（线上）"的占比
ch2 = '2' 
counts_pct2 = age_channel_counts_pct[2]   # 购入次数占比
sales_pct2 = age_channel_sales_pct[2]     # 销量占比
avg_pct2 = age_channel_avg_sales_pct[2]   # 人均消费占比

age_groups = counts_pct2.index.tolist()
x = np.arange(len(age_groups))
width = 0.25

HM_RED = '#E50914'

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, counts_pct2.values, width, label='购入次数占比', color='#F4A582')
bars2 = ax.bar(x,         sales_pct2.values,  width, label='销量占比',     color=HM_RED)
bars3 = ax.bar(x + width, avg_pct2.values,    width, label='人均消费占比', color='#92141C')

# 加数据标签
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h+0.005,
                f'{h:.1%}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(age_groups)
ax.set_title('不同年龄群体的线上渠道（渠道2）偏好对比', fontsize=15, fontweight='bold')
ax.set_xlabel('年龄', fontsize=12)
ax.set_ylabel('线上渠道占比', fontsize=12)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

三张分组柱状图显示，销售额占比、件数占比、人均消费在年龄×渠道维度上的结构高度一致，各群体均以线上渠道为主导。因此在后续偏好分析中采用销售额占比堆积图作为主要呈现

小结：
- 线上为主、线下为辅的结构在所有客群中保持稳定，群体间差异不超过 13 个百分点。这说明 H&M 的渠道布局对各年龄客群具有较强的普适性，渠道运营策略不需要按年龄群体差异化设计，重点应放在主渠道（线上）的体验优化和次渠道（线下）的功能定位上。

#### C. 年龄 × 品类偏好

In [ ]:
# 品类偏好
product_type_tran = (
    transactions[['customer_id', 'article_id']]
    .merge(customers[['customer_id', 'age_group']], on='customer_id', how='inner')
    .merge(articles[['article_id', 'product_type_name']], on='article_id', how='inner')
)
product_type_tran['product_type_name'].nunique()

In [ ]:
product_type_tran.shape

In [ ]:
# top_n 品类筛选标准：
# - 销量排名前 10% 左右（top_n_product 的 13 个品类合计占总销量约 80%）
# - 至少十万顾客购买过，覆盖主流消费
# - 13 个的数量既保证热力图可读性，又涵盖核心品类
top_n = 13
top_n_product = product_type_tran['product_type_name'].value_counts().head(top_n).index.tolist()
top_n_trans = product_type_tran[product_type_tran['product_type_name'].isin(top_n_product)]
top_n_counts = (
    top_n_trans
    .groupby(
        ['product_type_name', 'age_group'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(top_n_product)
)
top_counts_pct = top_n_counts.div(top_n_counts.sum(axis=0), axis=1)
top_counts_pct

In [ ]:
# 产品线偏好
# 由于index_name细节较多，这里选择index_group_name字段作为分析类别
product_index_tran = (
    transactions[['customer_id', 'article_id']]
    .merge(customers[['customer_id', 'age_group']], on='customer_id', how='inner')
    .merge(articles[['article_id', 'index_group_name']], on='article_id', how='inner')
)
print(product_index_tran.shape)
index_order = product_index_tran['index_group_name'].value_counts().index.tolist()
index_counts = product_index_tran.groupby(['index_group_name', 'age_group'], observed=True).size().unstack(fill_value=0)
index_counts = index_counts.reindex(index_order)
index_counts_pct = index_counts.div(index_counts.sum(axis=0), axis=1)
index_counts_pct

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

#品类
sns.heatmap(top_counts_pct, cmap='YlOrRd', annot=True, fmt='.1%', linewidth=0.5, ax=ax[0]
            , cbar_kws={'label': '购入次数占比', 'shrink':0.8}
            , annot_kws={'size':8, 'fontweight':'bold'}
           )
ax[0].set_title('不同客群对品类的消费偏好', fontsize=15)
ax[0].set_xlabel('客群', fontsize=12)
ax[0].set_ylabel('产品品类', fontsize=12)

#产品线
sns.heatmap(index_counts_pct, cmap='YlOrRd', annot=True, fmt='.1%', linewidth=0.5, ax=ax[1]
            , cbar_kws={'label': '购买量占比', 'shrink':0.8}
            , annot_kws={'size':8, 'fontweight':'bold'}
           )
ax[1].set_title('不同客群对产品线的消费偏好', fontsize=15)
ax[1].set_xlabel('客群', fontsize=12)
ax[1].set_ylabel('产品线', fontsize=12)

plt.tight_layout()
plt.show()

小结：
- 不同年龄群体对品类的偏好差异较小，其中trousers为所有群体共同购入频率最高的商品品类，建议推出更多样化的trousers以适应不同年龄群体的需求。
- 购入频率仅次于trousers的是dress，且图二中Ladieswear在各年龄群体中占据最大消费占比地位，共同表明H&M的主要顾客为女性，其中26-35岁群体购入dress的频次与trousers非常接近，建议在dress方面推出更符合年轻人喜好的风格。
- 在56岁及以上的人群中，品类偏好较为集中，除各群体购入频次都非常高的trousers、dress、sweater、T-shirt外，top和blouse的需求也很高，而其他群体的消费分布更为分散，表明大龄顾客对H&M服装的需求更多体现在功能性上例如保暖、便利，建议精准大龄顾客的需求。
- 在36-45岁的年龄群体中，婴幼儿及青少年服装的购入占比相对其他群体最大，建议在这类顾客下单时增加此类服装的推荐，提高交叉销售。

### 3.2.2 季节维度

#### A.  季节 × 品类偏好

In [ ]:
articles_top = articles[articles['product_type_name'].isin(top_n_product)][['article_id', 'product_type_name']]
season_type = (
    transactions[['article_id', 'season']]
    .merge(articles_top,on='article_id', how='inner')
              )

season_order = ['Spring', 'Summer', 'Fall', 'Winter']
season_type_counts = (
    season_type
    .groupby(['season', 'product_type_name'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(season_order)
)

season_type_pct = season_type_counts.div(season_type_counts.sum(axis=1), axis=0).T.reindex(top_n_product)
season_type_pct

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(season_type_pct, cmap='YlOrRd', annot=True, fmt='.1%', linewidths=0.5, ax=ax, 
            cbar_kws={'label':'销售占比', 'shrink':0.8}, 
            annot_kws={'size':8, 'fontweight':'bold'})
ax.set_title('各品类的季节性销售分布', fontsize=15)
ax.set_xlabel('季节', fontsize=12)
ax.set_ylabel('品类', fontsize=12)

plt.tight_layout()
plt.show()

小结：
- 秋季换季效应显著：sweater在秋季占比高达24.8%，呈现爆发式增长，可能受到提前活动例如双十一等影响，建议秋季营销资源向毛衣品类倾斜。
- 长裤地位稳固：trousers在四季保持14%以上的稳定占比，应加强其常年库存管理，保证供应稳定性。
- 隐形冬季社交需求：dress、skirt的销量在冬季仍保持较高占比（合计约17%），显示一定的冬季社交消费特征，建议增加保暖款的设计迎合冬季需求从而提高单价收入。
- 潜在清仓提醒：相比春夏，bikini top、swimwear bottom和shorts在秋冬季需求骤减，需要在夏季做好库存出清策略，避免跨季积压。

####  B. 时间 × 渠道分布

In [ ]:
season_channel = transactions[['year_month', 'sales_channel_id', 'estimated_price']]

monthly_sales = (
    season_channel
    .groupby(['year_month', 'sales_channel_id'])['estimated_price']
    .sum()
    .unstack(fill_value=0)
)/1000

monthly_counts = (
    season_channel
    .groupby(['year_month', 'sales_channel_id'])
    .size()
    .unstack(fill_value=0)
)

monthly_sales.index = monthly_sales.index.astype(str)
monthly_counts.index = monthly_counts.index.astype(str)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

#子图一：月销售额趋势
monthly_sales.plot(kind='line', ax=ax[0], marker='o', color=['#EF7B7E', '#E50000'])
ax[0].set_title('年月 × 渠道销售额趋势')
ax[0].set_xlabel('年月')
ax[0].set_ylabel('销售额（K）')
ax[0].grid(axis='y', alpha=0.3)
ax[0].set_ylim(bottom=0)

#子图二：月销量额趋势
monthly_counts.plot(kind='line', ax=ax[1], marker='o',color=['#EF7B7E', '#E50000'],)
ax[1].set_title('年月 × 渠道销量趋势')
ax[1].set_xlabel('年月')
ax[1].set_ylabel('销量（件）')
ax[1].grid(axis='y', alpha=0.3)
ax[1].set_ylim(bottom=0)

plt.tight_layout()
plt.show()

小结：
- 整体来看不同渠道每月的销售额与销量的波动方向一致；线上保持主要销售渠道地位，2020年初疫情爆发后线上销售增长符合疫情规律，几个月后随疫情情况改善线下逐渐恢复销售，线上销售有所回落。
- 2020年4月因欧洲疫情居家令影响，线下门店销售为零，线上作为唯一购物渠道迎来了销售占比巅峰，清晰展示了数字化布局在极端危机中的风险对冲能力。线上渠道的供应表现确保了企业的业务运转，避免了收入归零风险。
- 2020年5月起线下渠道表现出强劲的复苏韧性，销售迅速回升至疫情前水平，表明了实体门店在社交体验、试穿服务、即时获得感上的强烈不可替代地位。

### 3.2.3 渠道维度

#### A.  渠道 × 品类分布

In [ ]:
channel_type = (
    articles[articles['product_type_name']
        .isin(top_n_product)][['article_id', 'product_type_name']]
    .merge(
        transactions[['article_id', 'sales_channel_id', 'estimated_price']], 
        on='article_id', how='inner'
    )
)

channel_sale_counts = (
    channel_type
    .groupby('sales_channel_id', observed=True)
    .size()
)
channel_sale_counts_pct = channel_sale_counts.div(channel_sale_counts.sum(axis=0)).round(2)
print(channel_sale_counts_pct)

channel_type_counts = (
    channel_type
    .groupby(['product_type_name', 'sales_channel_id'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(top_n_product)
)
channel_type_counts_pct = channel_type_counts.div(channel_type_counts.sum(axis=1), axis=0).round(2)
channel_type_counts_pct

In [ ]:
channel_type.shape

小结：
- H&M线上销量占比约74%，将近线下的2.8倍。将商品按照品类细分后进行不同渠道的销售占比计算，可以观察不同品类在线上的销量占比波动不超过11%，因此不同品类的渠道分布与整体结构基本一致。在弱偏离范围内，泳装和裙装线上销量占比高于整体表现（线上81%-85%），而内裤、毛衣、T恤则相对较低（线上63%-66%）。
- 但即使是线下占比相对较高的品类，其线上销量也明显高于线下，整体上线上依然是 H&M 各品类的主导销售渠道。这种弱偏离不足以支持按品类做差异化的渠道运营，更适合作为线上商品视觉呈现优化（强化泳装/连衣裙的图片质量）和线下门店选品（适当增加内裤/毛衣陈列）的微调依据。

####  B. 渠道 × 价格差异

In [ ]:
channel_avg_price = (
    channel_type
    .groupby(['product_type_name', 'sales_channel_id'], observed=True)['estimated_price']
    .mean()
    .unstack(fill_value=0)
    .reindex(top_n_product)
    .round(2)
)
channel_avg_price['more_pct'] = ((channel_avg_price[2] - channel_avg_price[1]) / channel_avg_price[1]).round(2)

avg_more_pct = channel_avg_price['more_pct'].mean().round(2)
print(f'线上高于线下价格的平均比例：{avg_more_pct:.0%}')

channel_avg_price.style.format({'more_pct': '{:.0%}'})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

cap = channel_avg_price['more_pct'].sort_values(ascending=True)
colors = plt.cm.Reds(np.linspace(0.3, 0.9, 13))

barhs = cap.plot(
    kind='barh',
    ax=ax,
    color=colors,
    width=0.8,
    edgecolor='white'    
)

for i, val in enumerate(cap.values):
    ax.text(val+0.008, i, f'{val:.0%}', ha='center', va='center')
    
ax.set_title('不同品类线上价格高于线下的比例')
ax.set_xlabel('涨幅')
ax.set_ylabel('品类')
ax.xaxis.set_major_formatter(PercentFormatter(1.0)) 

plt.tight_layout()
plt.show()

小结：
- H&M销量最高的前13种品类无一例外地呈现"线上价格高于线下"的现象，涨幅在10%-34%之间。其中背心线上价格溢价最大（34%），其次是连衣裙、泳装与短袖（25%-27%）。
- 清仓压力：推测原因可能是线下门店仓储容量有限，强季节性单品（背心、连衣裙、泳装、短袖）过季后承受较大清仓压力，需经常打折促销，因此线下均价被拉低。而线上渠道仓储成本相对低，可以维持原价销售，这类品类的线上溢价比例就显得最大。 与之相反，裤子作为全年高频需求品类、不受季节限制，清仓压力较小，因此线上线下价格差异也最小（仅 10%）。
- 高端产品：在前面3.1.1中提过，H&M的部分高端系列更多选择在线上销售，因此在一定程度上拉高了线上渠道产品的均价。
- 购物习惯：线上渠道可能更多铺正价新款和当季产品，而线下保留较多前一季度的打折库存。线上的视觉营销和满减机制也更容易引导顾客挑选高客单价商品。

# 4. 客户价值与行为分析

## 4.1 RFM模型

In [ ]:
# 创建RFM表
last_date = transactions['t_dat'].max() + pd.Timedelta(days=1)
RFM = (
    transactions
    .groupby('customer_id')
    .agg(
        Recency = ('t_dat', 'max'),
        Frequency = ('t_dat', 'nunique'),
        Monetary = ('estimated_price', 'sum')
    )
    .reset_index()
)
RFM['Recency'] = (last_date - RFM['Recency']).dt.days

# 给RFM表赋分
label = [1, 2, 3, 4, 5]
RFM['R_score'] = pd.qcut(RFM['Recency'].rank(method='first', ascending=False), 5, labels=label)

for col in ['Frequency', 'Monetary']:
    RFM[f'{col[0]}_score'] = pd.qcut(RFM[col].rank(method='first'), 5, labels=label)

顾客分层

In [ ]:
RFM.set_index('customer_id', inplace=True)

# 为了把头部高价值客户单独识别出来这里选择大于中位数3作为标准
for col in ['R_score', 'F_score', 'M_score']:
    RFM[col[0]] = RFM[col].apply(lambda x: '高' if x>3 else '低')
    
RFM['RFM'] = RFM['R'] + RFM['F'] + RFM['M']

In [ ]:
segment_map = {
    '高高高': '重要价值客户',
    '低高高': '重要保持客户',
    '高低高': '重要发展客户',
    '低低高': '重要唤回客户',
    '高低低': '一般发展客户',
    '高高低': '一般价值客户',    
    '低高低': '一般保持客户',    
    '低低低': '一般挽留客户'    
}
RFM['label'] = RFM['RFM'].map(segment_map)

按顾客标签进行聚合计算并绘制树状图

In [ ]:
# 按顾客标签进行聚合计算
RFM = RFM.reset_index()
customer_segment = (
    RFM.groupby('label')
    .agg(
        count = ('customer_id', 'size'),
        purchase = ('Monetary', lambda x: x.sum()/1000),
        avg_purchase = ('Monetary', 'mean')
    )
    .round(2)
    .sort_values('avg_purchase', ascending=False)
)

# 对消费能力进行排名
customer_segment['purchasing_power_rnk'] = customer_segment['avg_purchase'].rank(ascending=False).astype(int)
customer_segment

In [ ]:
import squarify

fig, ax = plt.subplots(1, 2, figsize=(16, 8))

palette = sns.color_palette('Set3', len(customer_segment.index))
color_map = dict(zip(customer_segment.index, palette))
colors = [color_map[i] for i in customer_segment.index]

# 子图1：客群人数占比
sizes = customer_segment['count']
pcts = sizes/sizes.sum()
labels = [f'{idx}\n{pct:.0%}' for idx, pct in zip(customer_segment.index, pcts)] 

squarify.plot(ax=ax[0], sizes=sizes, label=labels, color=colors, 
              alpha=0.6, edgecolor='white', linewidth=2)

ax[0].set_title('H&M不同客群人数占比', fontsize=15)
ax[0].axis('off')

# 子图2：客群消费金额贡献占比
sizes = customer_segment['purchase']
pcts = sizes/sizes.sum()
labels = [f'{idx}\n{pct:.0%}' for idx, pct in zip(customer_segment.index, pcts)]

squarify.plot(ax=ax[1], sizes=sizes, label=labels, color=colors,
             alpha=0.6, edgecolor='white', linewidth=2)

ax[1].set_title('H&M各客群消费贡献占比', fontsize=15)
ax[1].axis('off')

plt.tight_layout()
plt.show()

顾客分层分析：
- 长尾效应明显（一般挽留客户最多，约 41%），“三低”客户占比将近一半，说明大量顾客只是偶然消费。
- 核心基本盘稳固（重要价值客户占比排名第二，约 22%），“三高”客户占比不低，说明 H&M 拥有一定的忠实高价值粉丝群体，建议倾斜更多资源，设计专享服务等，也可针对此类人群特供高附加值的商品。
- 一般发展客户&重要保持客户（高低低&低高高，均占比约10%），占比同样较高，需要增加广告营销、消费券推送等刺激再度消费。
- 流失预警（重要唤回客户占比约6%），这部分以前是高额消费人群，购买频次低且近期未消费，需要重点关注，可通过短信、邮件、微信公众号等推送方式建立挽回通道。

顾客价值分析：
- 22%的重要价值客户贡献了57%的价值，需要重点维护这类客户群体，强化留存策略，例如提供贵客专属权益、个性化优惠和推荐等。
- 重要保持客户（低高高）提供了第二大块收入，近期缺乏消费可能是缺少活动、优惠券或新品等刺激，若不加以措施可能为转换为流失客户，需要采取措施刺激消费或发放问卷了解情况。
- 占比最大的一般挽留客户（41%）仅贡献了10%的价值，需要施加关注，挖掘潜在客户，将更多的潜在收益转化为真实收入，可通过发送上新推送短信邮件、回归优惠券等福利召回。
- 重要发展客户消费能力排名靠前但总体人数较少，同样需要增强消费刺激，避免转化为重要挽留客户，使其更多转化为重要价值客户。

## 4.2 购物篮分析

由于没有订单编号，这里将同一天同一个顾客购买的所有商品视为同一条记录。

In [ ]:
df = (
    transactions[['t_dat', 'customer_id', 'article_id']]
    .merge(
        articles[['article_id', 'product_type_name']], 
        on='article_id'
    )
)
baskets = (
    df.drop_duplicates(['t_dat', 'customer_id', 'product_type_name'])
    .groupby(['customer_id', 't_dat'], observed=True)['product_type_name']
    .apply(list)
    .tolist()
)

# 为过滤无效篮子，只保留可能产生关联的多品类篮子（长度大于2）
baskets = [basket for basket in baskets if len(basket)>1]

In [ ]:
# 扫描一遍所有的购物篮（baskets），找出里面出现过的所有唯一品类并排序
# 并根据刚才的名单，把每一个购物篮转化成一行 True/False，再转换成DataFrame
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(baskets).transform(baskets)
df_baskets = pd.DataFrame(te_ary, columns=te.columns_)

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

# H&M品类多样、单次篮子小，取经验值0.01对应总交易量约1%的品类组合
frequent_itemsets = apriori(df_baskets, min_support=0.01, use_colnames=True)
rules = (
    association_rules(frequent_itemsets, metric='lift', min_threshold=1)
    .sort_values('lift', ascending=False)
)
print(rules['lift'].describe())
print(rules['confidence'].describe())

In [ ]:
# 根据数据分布与业界经验值，选定提升度1.2、置信值0.3作为筛选标准
final_rules = (
    rules[(rules['lift']>1.2) & (rules['confidence']>0.3)]
    [['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    .round(3)
)
final_rules.head(10)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
               rules['support'], rules['confidence'], 
               c=rules['lift'], s=rules['lift']*50,
               cmap='YlOrRd', alpha=0.6, edgecolor='gray'
           )

cbar=plt.colorbar(scatter)
cbar.set_label('Lift(提升度)', fontsize=12)

# 选择三种不同类型的强关联品类
target_rules = rules[
    ((rules['antecedents'] == frozenset(['Bikini top'])) & 
     (rules['consequents'] == frozenset(['Swimwear bottom']))) |
    ((rules['antecedents'] == frozenset(['Underwear bottom'])) & 
     (rules['consequents'] == frozenset(['Bra']))) |
    ((rules['antecedents'] == frozenset(['Cardigan'])) & 
     (rules['consequents'] == frozenset(['Sweater'])))
]

from adjustText import adjust_text

texts = []
for i, row in target_rules.iterrows():
    texts.append(plt.text(
        row['support'], row['confidence'],
        f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}",
        fontsize=9, fontweight='bold'
    ))

adjust_text(texts, arrowprops=dict(arrowstyle='->', color='gray'))

plt.title('H&M品类关联规则分布图', fontsize=15)
plt.xlabel('Support(支持度)', fontsize=12)
plt.ylabel('Confidence(置信度)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

左上角的高 lift 高 confidence 点群均为泳装相关衍生规则，与 Bikini top↔Swimwear bottom 表达同一洞察。

小结：
- 泳装上衣与下衣具有极强的双向关联（Lift=8.472），建议作为套装销售。
- 内裤对文胸的带动作用（49%）显著高于文胸对内裤的带动作用（39%）营销建议：在内裤的结算页面弹出文胸的加购提示，效果会比反过来更好。
- “跨品类”关联：Cardigan（针织开衫）带动Sweater（毛衣）的关联虽然不如本就为套装的泳装套装高，但它们展示了顾客的穿搭习惯。比如买开衫的人常买毛衣，可能说明他们喜欢叠穿。
- 此外，Jumpsuit/Playsuit 与 Dress 之间也存在中等强度关联（lift=1.81），反映出这两类客户群体高度重叠，均偏好"一件式"穿搭，可作为客户画像分析的补充参考。

## 4.3 留存分析

In [ ]:
cohort_df = transactions[['customer_id', 'year_month']].copy()
cohort_df['first_ym'] = cohort_df.groupby('customer_id')['year_month'].transform('min')
cohort_df['cohort_month'] = (cohort_df['year_month'] - cohort_df['first_ym']).apply(lambda x: x.n)

cohort = cohort_df.groupby(['first_ym', 'cohort_month'])['customer_id'].nunique().unstack(fill_value=0)
retention = cohort.divide(cohort.iloc[:, 0], axis=0).round(2)
retention

In [ ]:
# 留存热力图
plt.figure(figsize=(12,6))

# 把右下三角的"未来"区域置为 NaN
mask = np.zeros_like(retention, dtype=bool)
for i in range(len(retention)):
    mask[i, len(retention) - i:] = True

sns.heatmap(retention, annot=True, fmt='.0%', cmap='Reds', 
            vmin=0, vmax=0.6, mask=mask, cbar_kws={'label': '留存率'})

plt.title('H&M 顾客留存率热力图', fontsize=15)
plt.xlabel('留存月份', fontsize=12)
plt.ylabel('首购年月', fontsize=12)
plt.show()

小结：
- 2019-09在第 11 个月仍有 36% 留存，远高于2020-01的17%。可能原因：(1) 数据起点截断——2019-09 是数据起点，该月被识别为"首购"的用户实际很可能是更早就活跃的存量客户；(2) 早期获客质量更高（如品牌初期吸引的是核心粉丝群体）
- 次月留存：除2019年9月外，其余每月的次月留存都会流失68%-85%左右。次月是用户从"试用"到"复购"的关键转折点，建议通过给首次进店消费的顾客发放“次月复购券”等拉高留存率、提供每月积分兑换等方式培养顾客消费习惯；
- 季节回流：多个cohort 在相对月份4-6 个月处（即对应日历月 2020 年 4-6 月）出现留存率回升的小波峰（如 2019-12 cohort 第 5 月留存 23%、2020-01 cohort 第 5 月留存 21%）。该时段为欧美春夏换季高峰，叠加 H&M 季节促销，刺激了沉睡用户回流。建议在春夏换季前 1-2 周向沉睡用户（如 30 天未购）推送换季新品 + 限时折扣，主动激活回流意愿。

## 4.4 基于顾客分层的销售推荐

In [ ]:
# 筛选用于分析的商品列表
topn = 13
topn_list = (
    transactions
    .merge(articles[['article_id', 'product_type_name']], on='article_id')
    .groupby('product_type_name')
    .size()
    .sort_values(ascending=False)
    .index
    .to_list()[:topn]
)
topn_list

In [ ]:
# 顾客标签 X 商品名称 销量矩阵表
segment_type = (
    RFM[['customer_id', 'label']]
    .merge(transactions[['customer_id', 'article_id', 'estimated_price']], on='customer_id')
    .merge(
        articles[articles['product_type_name'].isin(topn_list)][['article_id', 'product_type_name']], 
        on='article_id')
)
print(segment_type.shape)
segment_type_counts = (
    segment_type.groupby(['label', 'product_type_name'], observed=True)
    .size()
    .unstack(fill_value=0)
)

In [ ]:
# 计算同一客群中不同商品的购买占比
purchase_pct = segment_type_counts.div(segment_type_counts.sum(axis=1), axis=0)

# 计算不同品类占总品类销量占比
type_pct = segment_type_counts.sum(axis=0)/segment_type_counts.sum().sum()

# 计算偏好矩阵
preference_index = purchase_pct.div(type_pct, axis=1).round(2)

# 排序
order = ['重要价值客户', '重要保持客户', '重要发展客户', '重要唤回客户', 
         '一般价值客户', '一般发展客户', '一般保持客户', '一般挽留客户']
preference_index = preference_index.reindex(order)
preference_index

In [ ]:
mask = (preference_index.T >= 0.8) & (preference_index.T <= 1.2)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    preference_index.T, cmap='RdBu_r', annot=True, fmt='.2f',
    center=1, linewidths=0.5, ax=ax, mask=mask,
    cbar_kws={'label': '偏好指数（仅显示 >1.2 或 <0.8）', 'shrink': 0.8},
    annot_kws={'size': 9, 'fontweight': 'bold'}
)

ax.set_title('各客群对品类的显著偏好', fontsize=15)
ax.set_facecolor('#f0f0f0')  # 灰色背景代表"无显著偏好"
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

小结：
- 重要价值客户 + 重要保持客户：这类顾客偏好均衡，无显著倾向，因此不需要做品类倾斜营销，重点在维护体验、提升复购频次。
-  重要发展客户 + 一般发展客户：存在泳装强偏好（Bikini top 1.47/1.47、Swimwear bottom 1.36/1.39），是季节性顾客。建议夏季前主动推送泳装新品 + 套装优惠。重要发展客户消费能力强但频次低，是泳装新品潜在买家。
-  低价值客群（一般价值/一般发展/一般保持 /一般挽留）：Underwear bottom 全员偏好（1.27-1.39）：这类顾客主要在H&M购买平价内裤，对Dress、Skirt等偏正式品类偏好都< 0.8。建议不要给该客群推送高客单价的连衣裙/西装裙，而应利用内裤作为引流品，结算时绑定 T-shirt（一般价值、一般发展也偏好T-shirt 1.30+）做满减组合
-  重要唤回顾客：那些购买频率较低且也有段时间未购买的顾客，从偏好矩阵看，她们倾向购买比基尼、泳装，属于季节性消费群体——上次购买后暂时没有新需求，导致沉睡。可在暑期前推送新款比基尼、泳装、短袖等商品宣传，精准激活。

# 5. 总结与建议

## 5.1 客户运营

- (1) 深入挖掘高净值银发市场：H&M的顾客中56岁及以上的大龄群体虽占少数，但相比年轻群体对单品价格的接受度更高、偏好高品质单品，且考虑到这些大龄群体对服装品类的偏好呈现出对保暖、便利功能的需要，建议针对大龄群体推出更高品质的服装（如羊毛外套、品质针织、剪裁西装等），契合其"低频但高客单"的购买模式，提高来自于大龄群体的消费收入。
- (2) 基于RFM的差异化运营：H&M占比约22%的三高用户贡献了约57%的营业收入，这部分顾客的消费模式呈现出对时尚风格的偏好，首要关注这类顾客的需求、提供高质的服务，将更多设计、服务等资源倾向于此类顾客，能有效提高资源利用效率；对于消费频率较低但消费金额较高、消费频率较高但近期未消费的顾客，投放营销短信、发放优惠券等广告，避免转换为流失顾客；对于占比较低的一般保持顾客（低高低）与一般价值顾客（高高低）则降低营销成本，维持被动触达，以优化整体ROI。
- (3) 捕捉季节性价格敏感机会：H&M存在大量对价格敏感的顾客，占比41%的一般挽留客户人均消费仅623元，需要利用好夏季大促、换季等节点，通过活动拉高消费的同时跟进库存清仓管理，避免旧款囤积，提高仓库利用效率。

## 5.2 产品策略

- (1) 核心品类保障：裤装作为H&M的全时段、全渠道“现金牛”需要保证供应，并不断探查市场需要与时兴风格，即时推出符合顾客喜好的产品，稳固市场地位。
- (2) 童装交叉销售策略：36-45岁群体的童装（Baby/Children）购买占比达 5.3%，是其他年龄段（1.0%-2.5%）的 2-5 倍，反映该群体大概率有学龄期及以下子女。在向该年龄段顾客推送适龄女装的同时，增加婴幼儿及青少年服装的曝光与组合优惠，实现"为自己+为孩子"的一站式购物，提升联单率。
- (3) 精准营销计划：根据年龄、消费习惯分析，可在夏季对年轻女性群体推送泳装套装、在冬季推送季节性裙装，刺激应季消费。
- (4) 子品牌线分层运营：Divided产品线在16-25岁群体占比达 27.4%（远高于66岁以上的15.2%），是年轻群体的主要兴趣点。建议加大Divided线的潮流款式投放、增加快速上新频次，巩固年轻群体心智；Ladieswear线则向46岁以上群体倾斜（占比 65.7%-75.9%），主打经典款、品质感设计。

## 5.3 销售战略

- (1) 数字化渠道战略作用：线上作为H&M主要销售渠道，在疫情爆发线下门店无法正常营业时起到了强劲的补充作用，随着数字化的推行，线上销售渠道将占据相当长时间不可撼动的地位，H&M应当做好库存管理，保证供应充足。H&M同时需要重视营销推广的作用，不断拓展线上销售平台，增加盈利来源。
- (2) 线下门店的价值回归：线下门店在疫情恢复后迅速复苏至疫情前的营业水平，证明了线下门店在提供顾客即时获得感、试穿服务上的不可替代性，需要做好“双线”管理，为应对未来可能出现的不确定性（如疫情、供应链中断等），需要建立线上线下双轨制的风险对冲机制。
- (3) 渠道差异化铺货：根据不同渠道的销售品类占比差异，线下应当确保裤装、T恤、毛衣的陈列区域面积与款式，提供给顾客充足的选择，利用触感优势刺激即时购买；线上保证裤装、裙装的视觉营销与推广供应，吸引顾客进店挑选。 

# 6.为Tableau看板导出聚合数据

In [ ]:
# 导出表用于制作Tableau看板的三张表
OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = customers.merge(RFM[['customer_id', 'label']], on='customer_id')
df.to_csv(OUTPUT_DIR/'Customers.csv', index=False, encoding='utf-8')

articles.to_csv(OUTPUT_DIR/'Articles.csv', index=False, encoding='utf-8')

transactions['weekday'] = transactions['t_dat'].dt.strftime('%a')
transactions['sales_channel_id'] = transactions['sales_channel_id'].map({1:'Offline', 2:'Online'})
transactions.to_csv(OUTPUT_DIR/'Transactions.csv', index=False, encoding='utf-8')

In [ ]:
# 由于四月线下销售额为零，制作并导出用于Tableau制作月销售额X渠道图&月客流X渠道图
monthly_channel = (
    transactions
    .groupby(['year_month', 'sales_channel_id'], observed=True)
    .agg({'estimated_price': 'sum',
          'customer_id': 'nunique'})
    .reset_index()    
)
monthly_channel.columns = ['year_month', 'sales_channel', 'total_revenue', 'total_customers']
monthly_channel['year_month'] = monthly_channel['year_month'].astype('str').replace('-', '.')

#补齐线下四月销售数据
monthly_channel.loc[len(monthly_channel)] = ['2020-04', 'Offline', 0, 0]
monthly_channel = monthly_channel.sort_values(['year_month', 'sales_channel']).reset_index(drop=True)

#导出
monthly_channel.to_csv(OUTPUT_DIR/'monthly_channel.csv', index=False)

In [ ]:
# 导出用于制作RFM*Product热力图的偏好矩阵
prefer_idx = preference_index.copy().stack().reset_index()
prefer_idx.columns = ['label', 'product_type_name', 'idx']
prefer_idx.to_csv(OUTPUT_DIR/'prefer_idx.csv', index=False)

In [ ]:
# 制作并导出购物篮子表
baskets = (
    transactions
    .groupby(['t_dat', 'weekday', 'customer_id', 'sales_channel_id'], observed=True)
    .agg(
        basket_price = ('estimated_price', 'sum'),
        product_count = ('article_id', 'count')
    ).reset_index()
)

baskets.columns = ['t_dat', 'weekday', 'customer_id', 'sales_channel', 'basket_price', 'product_count']

baskets['bc_id'] = baskets['t_dat'].astype(str) + '_' + baskets['customer_id'].astype(str)
baskets['bcs_id'] = (baskets['t_dat'].astype(str) + '_' + 
                     baskets['customer_id'].astype(str) + '_' + 
                     baskets['sales_channel'].astype(str))

baskets = baskets.sort_values('bcs_id')

baskets.to_csv(OUTPUT_DIR/'baskets.csv', index=False)